## Beginning of Task 1 -- Laurie

## Task 1a Part 1： Crude DP with Approximation, implementation with loop and Precomputation
### DP setup
1) State definitition: **(m,r)** at time **i**, where **m** is the current liquidity memory before submission, **r** is the remaining shares to be subimitted $$ r_i=N-\sum_{j=0}^{j=i-1} n_j $$.
2) The action and feasible action set: action at time **i** is the size of order submission **n_i**, feasible action set would be $$n_i<=r_i,\qquad n_i\in\mathbb Z_{\ge 0} $$
3) One step state transition function: from **i** to **i+1**, transition function for M is:
   $$ m_{i+1}=\left\lceil \frac{M_i+n_i}{2}\right\rceil $$
   transition funtion for **R** is
   $$ r_{i+1}=r_i-n_i $$
4) One-period reward:
   $$ reward_i=s_i=\left\lceil\bigl[1-\alpha M_i^\pi\bigr]n_i\right\rceil$$
5) Last period condition：at time **T**, all remaining orders should be submitted
$$ n_{T-1}=N-\sum_{i=0}^{T-2}n_i$$
equivalent to $$ r_{T-1}=n_{T-1}$$
6) Bellman recursion: Define **d(i,m,r)** as the maximum value-to-go at time interval **i**
$$ d(i,m,r)=\max_{n_i=0,\ldots,r_i}\left[s_i+d(i+1,m_{i+1},r_i-n_i)\right]$$

### Approximation and implementation in this part
1) Discretization

The algorithm define **n_m**,**n_r**,**n_n** as the number of equal space on **N** from 0 to 1000000.

Since **M** and **R** all have same order of magnitude as **N**，the initial values for the number of grids are **n_m**=100, **n_r**=100, and for discretization of actions (determining minimum unit of order sumbission), **n_n** is set to equal to **n_r** for simplicity.

The grid is not fine enoughwith stepsize around 1000, since implementation with loop is very time consuming, we fix the resolution issue with linear interpolation and finer grid in the **Part2** with vectorization.

The functiont **grid** produces **M_grid**, **R_grid** and **N_grid**

2) Bilinear Interpolation of value-to-go

When computing the value-to-go at time interval **i+1**, we need to update values of **M** and **R**, however, they are off the grid most of the time, so the stored value-to-go on the grid from previous interval in backward recursion appears insufficient.(In the simplified version however, with **n_n**=**n_r**, **R** value is always on the grid, so only interpolation on **M** is needed, defining **RegularGridInterpolator** is just for potential generalization purpose)

Suppose $$m_{lower}<=new\_m<=m_{upper}$$ $$r_{lower}<=new\_r<=r_{upper}$$ where the lower and upper values are both on the grid
Define the weights
$$ w_m = \frac{new\_m - m_{lower}}{m_{upper} - m_{lower}}, \quad w_r = \frac{new\_r - r_{lower}}{r_{upper} - r_{lower}} $$
therefore, the values of the four corners are
$$
\begin{aligned}
V_{LL} &= d(i+1, m_{lower}, r_{lower}) \\
V_{HL} &= d(i+1, m_{upper}, r_{lower}) \\
V_{LH} &= d(i+1, m_{lower}, r_{upper}) \\
V_{HH} &= d(i+1, m_{upper}, r_{upper})
\end{aligned}
$$
Therefore, we interpolate for the value across **M** and **R** under the function
$$
d_{interp}(i+1, new\_m, new\_r) = (1-w_m)(1-w_r)V_{LL} + w_m(1-w_r)V_{HL} + (1-w_m)w_r V_{LH} + w_m w_r V_{HH}
$$
Interpolation creates non-integer values for value-to-go function, in simulation, the filled sizes are rounded to integer

3) Precomputation

We precompute the values of **new_m** **new_r** and **reward(m,n)** and stored in the vector.

So that later given the current **m_i** and **r_i**'s indices in the grid, we looked up the matrix for the corresponding new values after different action **n_i**, and the reward value.

4) Implementation

The backward DP is naively implemented through loop, finding the value-to-go matrix, and then another function extracts the optimal schedule with a forward DP, to recover n for off-grid-values of **(m,r)**, bilinear interpolation is till necessary, candidates of **n** is taken from the N_grid.

Given the schedule, the execution is simulated with the exact formula without grid.

In [3]:
import csv
import sys
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
import time

In [4]:
from scipy.interpolate import RegularGridInterpolator
# Below is the baseline implementation with loop and without precomputation on Task 1a
# Define the DP function
# Define the discretization function
def grid(N,n_m,n_r,n_n): # n_m as the number of spaces for liquidity Memory, n_r as the number of spaces for remaining size,n_n as number of spaces for order submission:
    M_grid=np.arange(0,N+N/n_m,N/n_m)
    R_grid=np.arange(0,N+N/n_r,N/n_r)
    N_grid=np.arange(0,N+N/n_n,N/n_n)
    return M_grid,R_grid,N_grid
# Define the transition function of M（original without precomputation)
def transition(m,n):
    m=math.ceil((m+n)/2)
    return m
# Define the reward function (original without precomputation)
def reward(n,new_m,alpha,pi):
    return math.ceil(max(0.0,1.0-alpha*new_m**pi)*n)

def DP_loop(N,T,pi,alpha,d:np.array):
    M_grid,R_grid,N_grid=grid(N,n_m,n_r,n_n)
    # Precomputation of the value of new_m, reward, and new_r
    new_m_mtrx=np.ceil((M_grid[:,None]+N_grid[None,:])/2) # n_m*n_n
    reward_mtrx=np.ceil(np.maximum(0,1-alpha*new_m_mtrx**pi)*N_grid[None,:]) # n_m*n_n
    new_r_mtrx=R_grid[:,None]-N_grid[None,:] #n_r*n_n, the new remaining shares value when n shares submitted, for all values on N_grid
    # Define the terminal Condition (submitting all remaining orders at T-1)
    for r_idx,r in enumerate(R_grid) :
        n=r
        for m_idx,m in enumerate(M_grid):
            new_m=new_m_mtrx[m_idx,r_idx] #Here since n=r and N and R's grid n_n=n_r, n_idx=r_idx
            d[T-1,m_idx,r_idx]=reward_mtrx[m_idx,r_idx]
    print(f't={T} completed')
    # Backward recursion
    for i in range(T-2,-1,-1):
        #Define interpolator for the period i+1
        interp = RegularGridInterpolator((M_grid, R_grid), d[i+1,:,:],bounds_error=False, fill_value=-np.inf)
        # loop over states (m,r)
        for r_idx,r in enumerate(R_grid):
            for m_idx,m in enumerate(M_grid):
                if r==0:
                    d[i,m_idx,r_idx]=0
                else:
                    best=-np.inf
                    for n_idx,n in enumerate(N_grid):
                        if n>r:
                            break
                        # Compute the new liquidity memory value after order submisssion
                        new_m=new_m_mtrx[m_idx,n_idx]
                        new_r=new_r_mtrx[r_idx,n_idx]
                        # finding the next period value-to-go
                        if new_r==0: # interpolating only across grid of M
                            filled_later=np.interp(new_m,M_grid,d[i+1,:,0])
                        else:
                            # i+1 period's value-to-go is of the grid, interpolate
                            filled_later=interp([[new_m,new_r]])[0]
                        filled_now=reward_mtrx[m_idx,n_idx]
                        value=filled_now+filled_later
                        if value>best:
                            best=value
                    d[i,m_idx,r_idx]=best
        print(f't={i+1} completed')
    return d,M_grid,R_grid,N_grid

# Extract optimal schedule
def extract_schedule(d,N,T,alpha,pi,M_grid,R_grid,N_grid):
    schedule=[]
    m=0
    r=N
    for i in range(T):
        # Optimal action at T is to submit every shares left
        if i==T-1:
            n=r
            best_n=n
        else:
            interp = RegularGridInterpolator((M_grid, R_grid), d[i+1,:,:],bounds_error=False, fill_value=-np.inf)
            # if already all are submitted, n_i=0
            if r==0:
                n=0
                best_n=0
            else:
                best=-np.inf
                best_n=0
                for n_idx,n in enumerate(N_grid):
                    if n>r:
                        break
                    else:
                        new_m=transition(m,n)
                        new_r=r-n
                        if new_r==0:
                            filled_later=np.interp(new_m,M_grid,d[i+1,:,0])
                        else:
                            filled_later=interp([[new_m,new_r]])[0]
                        value=reward(n,new_m,alpha,pi)+filled_later
                        if value>best:
                            best=value
                            best_n=n
        schedule.append(best_n)
        m=transition(m,best_n)
        r=r-best_n
    return np.array(schedule,dtype=int)
def simulate_schedule(schedule, alpha, pi):
    """Simulate the liquidity-memory model for a given integer order schedule."""
    M = 0
    fills = []
    memory = []

    for n_i in schedule:
        n_i = int(n_i)
        M = math.ceil(0.5 * (M + n_i))
        traded = math.ceil(max(0.0, 1.0 - alpha * M**pi) * n_i)

        memory.append(M)
        fills.append(traded)

    return np.asarray(fills, dtype=int), np.asarray(memory, dtype=int)

In [5]:
# Implementation
t0=time.perf_counter()
N=100000
T=10
alpha=0.001
pi=0.5
n_m=100
n_r=100
n_n=100
d=np.full((T,n_m+1,n_r+1),-np.inf)
d,M_grid,R_grid,N_grid=DP_loop(N,T,pi,alpha,d)
schedule_app_loop=extract_schedule(d,N,T,alpha,pi,M_grid,R_grid,N_grid)
fills_app, memory_app = simulate_schedule(schedule_app_loop, alpha, pi)

for i in range(T):
    print(f"period {i}: order = {schedule_app_loop[i]:5d}, M = {memory_app[i]}, traded = {fills_app[i]}")
print("total traded =", fills_app.sum())
print(f'total runtime:{(time.perf_counter()-t0)}')

t=10 completed
t=9 completed
t=8 completed
t=7 completed
t=6 completed
t=5 completed
t=4 completed
t=3 completed
t=2 completed
t=1 completed
period 0: order = 12000, M = 6000, traded = 11071
period 1: order = 10000, M = 8000, traded = 9106
period 2: order =  9000, M = 8500, traded = 8171
period 3: order =  9000, M = 8750, traded = 8159
period 4: order =  9000, M = 8875, traded = 8153
period 5: order =  9000, M = 8938, traded = 8150
period 6: order =  9000, M = 8969, traded = 8148
period 7: order =  9000, M = 8985, traded = 8147
period 8: order = 10000, M = 9493, traded = 9026
period 9: order = 14000, M = 11747, traded = 12483
total traded = 90614
total runtime:313.3865411679999


### Results stated above

## Beginning of Vectorization -- Grace

**Part 2 (this notebook's new contribution):**
1. A **vectorized** backward DP. The vectorized implementation replaces the Python loops over liquidity-memory and remaining-inventory states with NumPy array operations, retaining loops over time periods and feasible actions.
2. A **correctness check**: vectorized vs. loop on identical grids (value tables and schedules must match).
3. A **runtime comparison**: loop vs. vectorized.
4. A **refinement check**: finer grids, with objective value, schedule and runtime reported at each level.

### DP setup (recap)
* **State** at time $i$: $(m, r)$. $m$ is the liquidity memory before submission, $r_i = N-\sum_{j<i} n_j$ is the remaining shares.
* **Action**: $n_i \in \{0,\dots,r_i\}$.
* **Transition**: $m_{i+1}=\lceil (m_i+n_i)/2\rceil$, $r_{i+1}=r_i-n_i$.
* **Reward**: $s_i=\lceil [1-\alpha\, m_{i+1}^{\pi}]_+\, n_i\rceil$ (the memory in the reward is the *updated* memory $M_i=\lceil (M_{i-1}+n_i)/2\rceil$).
* **Last period**: $n_{T-1}=r_{T-1}$.
* **Bellman**: $d(i,m,r)=\max_{0\le n\le r}\big[s(m,n)+d(i+1,m',r-n)\big]$, with $d(T-1,m,r)=s(m,r)$ and $d(\cdot,m,0)=0$.

**Backward DP.** For a fixed action index $k$ (order size $n=k\Delta$), the states with $r_{idx}\ge k$ move to $r_{idx}-k$. Writing the interpolation as *lo[m,k]* (lower $M$-grid index) and *w[m,k]* (weight on the upper point):

$$\text{cont}[m,\,r_{idx}] = (1-w_{m,k})\,V[lo_{m,k},\,r_{idx}-k] + w_{m,k}\,V[lo_{m,k}+1,\,r_{idx}-k],$$

computed for **all** $m$ and all $r_{idx}\in\{k,\dots,n_R\}$ in one shot with fancy indexing. The Python loop is kept only over $T$ and over the $n_r+1$ actions.

**Forward extraction.** Similar idea -- for the single actual state $(m, r)$, evaluate every feasible action at once. *np.argmax* returns the first maximizer, which matches the loop's strict *>* rule (smallest $n$ among ties).

### Precompute

**Purpose:** calculates the three following things that the approximate DP would otherwise keep recalculating

*   *new_m*: next period liquidity memory / determines the next state and fill rate
*   *rew*: number of shares filled for each memory or action combination / provides the immediate reward
*   *lo, w*: lower grid index and interpolation weight; approximate future DP values when the next memory falls between grid points


In [6]:
def precompute(N, alpha, pi, n_m, n_r):
    # time-invariant objects: next-memory, reward, interpolation index/weight (all shape nM x nK)
    n_n = n_r # set the number of action-grid intervals to the amount of remaining inventory grid intervals
    # this ensures that the action grid and inventory grid have the same spacing
    M_grid, R_grid, N_grid = grid(N, n_m, n_r, n_n)
    # creates three grids (liquidity memory, remaining-inventory, and possible order sizes/actions)
    nM = len(M_grid)
    # number of memory point grids
    ds = N / n_m
    # calculate spacing between consecutive memory-grid points
    new_m = np.ceil((M_grid[:, None] + N_grid[None, :]) / 2)
    # calculate the next-period liquidity memory for every combination of current memory and order size
    # m_next is ceiling of halfway between: ceil(m+n/2)
    # [: None] turns the M_grid into a column vector
    # [None: ] turns the N_grid into a row vector
    rew = np.ceil(np.maximum(0.0, 1.0 - alpha * new_m**pi) * N_grid[None, :])
    # calculate the number of shares actually filled for each (m,n) combination
    # fills = ceil(max(0, 1 - alpha * m_next**pi) * n); prevents the fill from becoming negative
    # the new_m**pi calculates the nonlinear liquidity-memory impact
    # the alpha value controls the impact
    # multiplying by N_grid gives the number of shares filled
    # round up to nearest integer
    lo = np.minimum(np.floor(new_m / ds).astype(np.int64), nM - 2) # find the index of the lower memory-grid point for each next-memory value
    # new_m/ds expresses the next-memory value in grid-index units
    # np.floor takes the lower grid index
    # np.minimum(..., nM - 2) ensures that an upper neighboring grid point (lo+1) always exists for interpolation
    w = (new_m - M_grid[lo]) / ds
    # calculate the linear interpolation weight on the upper memory-grid point
    # M_grid[lo] is the lower neighboring grid value
    # the numerator measures how far new_m lies above that lower point
    # divide by ds to find what fraction of the grid space this distance is
    # weight on the lower point is 1-w
    return M_grid, R_grid, N_grid, rew, lo, w
    # grids and precomputed arrays are returned thru out the dynamic programming algo

In [7]:
def DP_vec(N, T, pi, alpha, n_m, n_r):
    # implementing the vectorized approximate DP algorithm
    # N = total number of shares
    # T = total number of trading periods
    # pi = exponent that controls the effect of liquidity memory
    # alpha = parameter that controls strength of market impact
    # n_m = number of market grid intervals
    # n_r = number of remaining inventory grid algorithms
    M_grid, R_grid, N_grid, rew, lo, w = precompute(N, alpha, pi, n_m, n_r)
    # call the precompute function to get the following:
    # M_grid = possible liquidity-memory states
    # R_grid = possible remaining-inventory states
    # N_grid = possible order sizes (actions)
    # rew = immediate shares filled for each (memory, action) combination
    # lo = index of the lower memory-grid point for interpolation
    # w = interpolation weight on the upper memory-grid point
    # these are calculated once and reused thru out the DP

    nM, nR = len(M_grid), len(R_grid)
    # count the number of grid points for memory and remaining inventory
    # these determine the dimensions of the DP value table

    d = np.full((T, nM, nR), -np.inf)
    # initialize a 3D DP value table with negative infinity
    # dimensions: (time period, memory state, remaining inventory)
    # d[i, m_idx, r_idx] represents the maximum estimated number of shares that can be filled from period i onward, starting at
    # the memory and inventory states corresponding to these indices
    # negative infinity indicates that a state has not yet been evaluated
    d[T - 1] = rew
    # set the terminal condition for the final trading period
    # in the last period, all remaining shares must be submitted.
    # therefore, the chosen action n equals remaining inventory r
    # since N_grid and R_grid are identical, action index = inventory index
    # rew[:, r_idx] gives the actual filled shares when submitting all remaining inventory at each possible memory state
    # no future rewards need to be considered in the final period

    for i in range(T - 2, -1, -1):
    # perform backward induction from the second-to-last period to period 0
    # we work backward because the optimal decision at period i depends on the future value function at period i+1
        V = d[i + 1]
        # extract the DP value table for the following period.
        # V has shape (nM, nR)
        # these values represent the maximum estimated future shares filled for every combination of next-period memory and remaining inventory
        best = np.full((nM, nR), -np.inf)
        # initialize the best achievable value at each current state
        # for each combination of memory and remaining inventory, we will compare all feasible actions and retain the maximum
        # initially, no candidate actions have been evaluated
        for k in range(nR):
          # iterate through all possible action-grid indices
            lk, wk = lo[:, k], w[:, k]
            # extract interpolation information for action k
            # lk contains lower memory-grid indices for every current memory state
            # wk contains the corresponding upper-grid interpolation weights
            # the next memory depends on current memory and action, but not on the remaining-inventory state
            cont = (1 - wk)[:, None] * V[lk, :nR - k] + wk[:, None] * V[lk + 1, :nR - k]
            # calculate the estimated future value after submitting action k
            # the next memory may lie between two grid points, so we use linear interpolation:
            # future value = (1-w)*V(lower memory) + w*V(upper memory)
            # V[lk, :nR-k] selects future values at lower memory-grid points
            # V[lk+1, :nR-k] selects future values at upper memory-grid points
            # the future remaining inventory is r - n; since the grids are equally spaced, its index is r_idx - k
            # the slice :nR-k covers all possible future inventory indices
            val = rew[:, k][:, None] + cont
            # this is to calculate the total candidate value for choosing action k
            # Bellman equation: Candidate value = immediate reward + future continuation value
            # rew[:, k] contains the shares immediately filled at each memory state
            # [:, None] converts the reward vector into a column vector, allowing it to be added to all feasible inventory states
            # val has shape (nM, nR-k)

            np.maximum(best[:, k:], val, out=best[:, k:])
            # compare the candidate value with the best value found so far
            # best[:, k:] selects all current inventory states where action
            # k is feasible (i.e., remaining inventory >= submitted order size)
            # np.maximum() calculates the element-wise maximum
            # out=best[:, k:] updates the existing array directly, avoiding the need to create another output array
            # after considering all k, best stores the maximum Bellman value for every memory and remaining-inventory state
        best[:, 0] = 0.0
        # set the value for zero remaining inventory to zero
        # if r = 0, then there is nothing left to trade
        # therefore, the maximum future number of shares filled is zero, regardless of the current liquidity-memory state
        d[i] = best
        # save the optimized value table for period i
        # these values will be used as future continuation values when evaluating decisions in the previous peri
    return d, M_grid, R_grid, N_grid
    # return the complete DP value table and the three discretized grids
    # d contains the approximate optimal value for each time and state
    # the grids identify the actual state values associated with each index
    # these outputs can be passed to extract_schedule() to recover the approximate optimal sequence of order sizes

In [8]:
def extract_vec(d, N, T, alpha, pi, M_grid, R_grid, N_grid):
    # extract the approximate optimal trading schedule from the value table calculated by DP_vec()
    nM = len(M_grid) # number of memory grid points (make sure the indices stay within range)
    ds = M_grid[1] - M_grid[0]
    # calculate the spacing between consecutive memory-grid points
    # we assume the memory grid is uniformly spaced
    sched, m, r_idx = [], 0, len(R_grid) - 1
    # sched: empty list to store the selected order size for each period
    # m = 0: initial liquidity memory, meaning no previous trading impact
    # r_idx: index of the initial remaining inventory N
    # since R_grid ends at N, its last index corresponds to all N shares remaining
    for i in range(T):
        # move forward through trading periods 0, 1, ..., T-1
        # unlike DP_vec(), which works backward to calculate optimal values, this function works FORWARD to recover the trading decision
        if i == T - 1:
            k = r_idx
            # during the last trading period, just submit all remaining shares
            # since N_grid and R_grid are identical, the action index equals the remaining-inventory index
        elif r_idx == 0:
            k = 0
            # there is no inventory left to submit, so no additional shares can be filled in this or subsequent periods
        else:
            ks = np.arange(r_idx + 1)
            # generate indices for all feasible actions
            # since we cannot submit more shares than remain, the action indices range from 0 through r_idx
            nn = N_grid[ks]
            # nn contains all feasible numbers of shares we could submit in the current period
            nm = np.ceil((m + nn) / 2)
            # calculate the next liquidity-memory value for every action: m_next = ceil((m + n) / 2)
            # m is the actual current memory state
            # nn contains all possible order sizes
            lo = np.minimum(np.floor(nm / ds).astype(np.int64), nM - 2)
            # find the lower memory-grid index for each next-memory value
            ww = (nm - M_grid[lo]) / ds
            # callculate the interpolation weight for the upper grid point
            # M_grid[lo] contains the lower neighboring memory values
            # nm - M_grid[lo] measures how far each next-memory value lies above its lower neighboring grid point
            # the lower grid point receives weight (1-ww)
            # the upper grid point receives weight ww
            V = d[i + 1]
            # V has dimensions (memory state, remaining inventory)
            # the values represent the maximum estimated future shares filled starting from each possible next-period state
            cont = (1 - ww) * V[lo, r_idx - ks] + ww * V[lo + 1, r_idx - ks]
            # the next remaining inventory is r - n
            # since inventory and action grids have identical spacing, the next inventory index is r_idx - ks
            # V[lo, r_idx-ks] retrieves future values at the lower neighboring memory-grid points
            # V[lo+1, r_idx-ks] retrieves future values at the upper neighboring memory-grid points

            # linear interpolation combines these neighboring values: Continuation value = (1-ww)*V(lower) + ww*V(upper)

            rw = np.ceil(np.maximum(0.0, 1.0 - alpha * nm**pi) * nn)
            # calculate the immediate number of shares filled for each feasible order size
            # the reward function is: filled = ceil(max(0, 1 - alpha*m_next**pi) * n)
            k = int(np.argmax(rw + cont))
            # select the action that maximizes immediate reward plus estimated future reward
            # this implements the Bellman decision rule: optimal action = argmax(immediate fills + continuation value)
            # because ks = [0, 1, ..., r_idx], this index is also the corresponding action-grid index
        sched.append(int(N_grid[k]))
        # add the chosen order size to the trading schedule
        # N_grid[k] converts the selected action index into actual shares
        # the schedule records submitted shares, not necessarily filled shares
        m = math.ceil((m + N_grid[k]) / 2)
        # update the actual liquidity-memory state after submitting the order
        # the transition is m_next = ceil((m + n) / 2)
        r_idx -= k
        # since action and inventory grids have identical spacing, subtracting action index k gives the new inventory index
        # inventory decreases by submitted order size, not filled shares
    return np.array(sched, dtype=int)
    # the returned array has T elements, one for each trading period
    # it represents the approximate optimal sequence of submitted orders

In [9]:
def solve_vec(N, T, pi, alpha, n_m, n_r):
    # this runs the full pipeline: solve the DP via backwards induction, extract the approx optimal trading schedule, simulate the schedule to calculate
    # the actually filled shares, and then return the results and runtime measurements
    t0 = time.perf_counter()
    d, Mg, Rg, Ng = DP_vec(N, T, pi, alpha, n_m, n_r)
    t_dp = time.perf_counter() - t0
    sched = extract_vec(d, N, T, alpha, pi, Mg, Rg, Ng)
    t_tot = time.perf_counter() - t0 # get runtime
    fills, mem = simulate_schedule(sched, alpha, pi)
    # fills: actual shares filled in each trading period
    # mem: liquidity-memory values generated during simulation
    # this evaluates the schedule without interpolating DP values
    return dict(d=d, schedule=sched, fills=fills, memory=mem, total=int(fills.sum()),
                dp_estimate=float(d[0, 0, -1]), t_dp=t_dp, t_total=t_tot)

## Correctness Check (check vectorized against loop results)

The loop version is slow, so the full value-table comparison is done on small grids ($n=20$ and $n=40$) with the real problem parameters ($N=100000,\ T=10,\ \alpha=0.001,\ \pi=0.5$). This checks that (i) the value tables agree to a floating-point precision, (ii) the extracted schedules coincide, (iii) the simulated totals coincide with the loop results that Laurie derived.

In [10]:
def grid(N, n_m, n_r, n_n):
    # create evenly spaced grids for liquidity memory, remaining inventory, and order size
    M_grid = np.arange(n_m + 1) * (N / n_m)
    # generate n_m + 1 memory values from 0 to N
    R_grid = np.arange(n_r + 1) * (N / n_r)
    # generate n_r + 1 remaining-inventory values from 0 to N
    N_grid = np.arange(n_n + 1) * (N / n_n)
    # generate n_n + 1 possible order sizes from 0 to N
    return M_grid, R_grid, N_grid
    # return all three grids for use in the approximate DP

# THE FOLLOWING LOOP IS BASED ON LAURIE'S CODE
def DP_loop(N, T, pi, alpha, n_m, n_r, n_n, verbose=False):
    # implement the loop-based approximate DP using backward induction
    M_grid, R_grid, N_grid = grid(N, n_m, n_r, n_n)
    # generate the discretized state and action grids
    d = np.full((T, n_m + 1, n_r + 1), -np.inf)
    # initialize the DP value table with negative infinity for all periods and states
    # precompute state transitions and immediate rewards
    new_m_mtrx = np.ceil((M_grid[:, None] + N_grid[None, :]) / 2)
    # calculate next-period memory m' = ceil((m+n)/2) for every memory/action pair
    reward_mtrx = np.ceil(np.maximum(0, 1 - alpha * new_m_mtrx**pi) * N_grid[None, :])
    # calculate the immediate shares filled for every memory/action pair
    new_r_mtrx = R_grid[:, None] - N_grid[None, :]
    # calculate remaining inventory r' = r-n for every inventory/action pair
    # terminal condition: submit all remaining inventory in the final period
    for r_idx, r in enumerate(R_grid):
        # iterate through every possible remaining-inventory state
        for m_idx, m in enumerate(M_grid):
            # iterate through every possible liquidity-memory state
            d[T - 1, m_idx, r_idx] = reward_mtrx[m_idx, r_idx]
            # store final-period fills, assuming R_grid and N_grid are identical
    # backward recursion: calculate optimal values from T-2 to period 0
    for i in range(T - 2, -1, -1):
        # work backward because current optimal decisions depend on future values
        interp = RegularGridInterpolator((M_grid, R_grid), d[i + 1, :, :],
                                         bounds_error=False, fill_value=-np.inf)
        # create an interpolator to estimate future DP values at off-grid states
        # assign negative infinity to states outside the grid boundaries
        for r_idx, r in enumerate(R_grid):
            # evaluate every possible remaining-inventory state
            for m_idx, m in enumerate(M_grid):
                # evaluate every possible liquidity-memory state
                if r == 0:
                    # if no inventory remains, no additional shares can be filled
                    d[i, m_idx, r_idx] = 0
                    # set the optimal future reward to zero
                else:
                    best = -np.inf
                    # initialize the best candidate value before testing actions
                    for n_idx, n in enumerate(N_grid):
                        # evaluate possible order sizes in ascending order
                        if n > r:
                            break
                            # stop once the order size exceeds remaining inventory
                        new_m = new_m_mtrx[m_idx, n_idx]
                        # retrieve the precomputed next-memory state for action n
                        new_r = new_r_mtrx[r_idx, n_idx]
                        # retrieve the remaining inventory after submitting n shares
                        if new_r == 0:
                            # handle the case where all remaining inventory is submitted
                            filled_later = np.interp(new_m, M_grid, d[i + 1, :, 0])
                            # interpolate the future value along memory with zero inventory
                        else:
                            filled_later = interp([[new_m, new_r]])[0]
                            # interpolate the continuation value at the next state (m', r')
                        value = reward_mtrx[m_idx, n_idx] + filled_later
                        # calculate candidate value as immediate fills plus estimated future fills
                        if value > best:
                            # check whether this action improves the current best value
                            best = value
                            # update the maximum achievable value for this state
                    d[i, m_idx, r_idx] = best
                    # store the best value across all feasible actions
        if verbose:
            # print progress if verbose mode is enabled
            print(f"t={i+1} completed")
            # display the completed trading period
    return d, M_grid, R_grid, N_grid
    # return the approximate DP value table and the three grids

In [11]:

# set the initial number of shares, trading periods, and market impact parameters
N, T, alpha, pi = 100_000, 10, 0.001, 0.5
# initialize an empty list to store validation and runtime comparison results
check_rows = []
# test both implementations using grid resolutions of 20 and 40
for n in [20, 40]:
    # start timing the original loop-based implementation
    t0 = time.perf_counter()
    # run the original approximate DP using n grid intervals for each variable
    d_l, Mg, Rg, Ng = DP_loop(N, T, pi, alpha, n, n, n)
    # extract the approximate optimal trading schedule from the loop-based DP
    s_l = extract_schedule(d_l, N, T, alpha, pi, Mg, Rg, Ng)
    # calculate the combined runtime of the loop-based DP and schedule extraction
    t_loop = time.perf_counter() - t0
    # start timing the vectorized implementation
    t0 = time.perf_counter()
    # run the vectorized DP using the same parameters and grid resolution
    d_v, Mg2, Rg2, Ng2 = DP_vec(N, T, pi, alpha, n, n)
    # extract the approximate optimal trading schedule from the vectorized DP
    s_v = extract_vec(d_v, N, T, alpha, pi, Mg2, Rg2, Ng2)
    # calculate the combined runtime of the vectorized DP and schedule extraction
    t_vec = time.perf_counter() - t0
    # simulate the loop-based schedule and calculate total shares filled
    tot_l = simulate_schedule(s_l, alpha, pi)[0].sum()
    # simulate the vectorized schedule and calculate total shares filled
    tot_v = simulate_schedule(s_v, alpha, pi)[0].sum()
    # store the comparison results for the current grid resolution
    # check whether value tables and schedules match and measure runtime improvement
    check_rows.append(dict(n=n, tables_match=np.allclose(d_l, d_v),
                           max_abs_diff=np.max(np.abs(d_l - d_v)),
                           schedules_match=bool(np.array_equal(s_l, s_v)),
                           total_loop=tot_l, total_vec=tot_v,
                           t_loop_s=round(t_loop, 2), t_vec_s=round(t_vec, 3),
                           speedup=round(t_loop / t_vec, 1)))
# display the comparison results as a pandas DataFrame
pd.DataFrame(check_rows)

,n,tables_match,max_abs_diff,schedules_match,total_loop,total_vec,t_loop_s,t_vec_s,speedup
0,20,True,0.0,True,90567,90567,2.59,0.006,462.1
1,40,True,0.0,True,90595,90595,20.85,0.019,1123.5


## Baseline at $n_m=n_r=n_n=100$: loop vs. vectorized
Laurie's loop implementation took **495.8 s** at this grid size (when run in her notebook). Set RERUN_LOOP = True to reproduce it. The vectorized version is run live and compared to her recorded schedule and total.

In [12]:

# determine whether to rerun the original loop-based DP or use recorded results
RERUN_LOOP = True
# store the partner's previously recorded results at grid resolution 100
loop_schedule_recorded = np.array([12000, 10000, 9000, 9000, 9000, 9000, 9000, 9000, 10000, 14000])
# record the total number of shares filled by the original loop-based schedule
loop_total_recorded = 90614
# record the original loop implementation's runtime in seconds
loop_time_recorded = 495.818411400076
# rerun the original loop-based DP if RERUN_LOOP is True
if RERUN_LOOP:
    # start timing the loop-based implementation
    t0 = time.perf_counter()
    # compute the approximate DP using the original loops at grid resolution 100
    d_l, Mg, Rg, Ng = DP_loop(N, T, pi, alpha, 100, 100, 100)
    # extract the optimal trading schedule from the loop-based DP
    loop_schedule = extract_schedule(d_l, N, T, alpha, pi, Mg, Rg, Ng)
    # calculate the combined runtime of DP computation and schedule extraction
    loop_time = time.perf_counter() - t0
    # simulate the extracted schedule and calculate the total shares filled
    loop_total = int(simulate_schedule(loop_schedule, alpha, pi)[0].sum())
else:
    # use the partner's recorded schedule, objective value, and runtime without rerunning
    loop_schedule, loop_total, loop_time = loop_schedule_recorded, loop_total_recorded, loop_time_recorded
# run the vectorized implementation at the same grid resolution of 100
res100 = solve_vec(N, T, pi, alpha, 100, 100)
# display the original loop schedule, total shares filled, and runtime
print("loop schedule      :", loop_schedule.tolist(), "| total =", loop_total, f"| time = {loop_time:.1f}s")
# display the vectorized schedule, total shares filled, and runtime
print("vectorized schedule:", res100['schedule'].tolist(), "| total =", res100['total'], f"| time = {res100['t_total']:.3f}s")
# check whether both implementations produce identical trading schedules
print("schedules identical:", np.array_equal(loop_schedule, res100['schedule']))
# calculate the speedup factor by dividing loop runtime by vectorized runtime
print(f"speedup: {loop_time / res100['t_total']:.0f}x")
# retrieve the simulated fills and liquidity-memory values from the vectorized results
fills, mem = res100['fills'], res100['memory']
# display the submitted order, liquidity memory, and actual shares filled for each period
for i in range(T):
    # print the trading results for the current period
    print(f"period {i}: order = {res100['schedule'][i]:6d}, M = {mem[i]:6d}, traded = {fills[i]:6d}")
# display the total number of shares actually traded across all periods
print("total traded =", fills.sum())

loop schedule      : [12000, 10000, 9000, 9000, 9000, 9000, 9000, 9000, 10000, 14000] | total = 90614 | time = 310.4s
vectorized schedule: [12000, 10000, 9000, 9000, 9000, 9000, 9000, 9000, 10000, 14000] | total = 90614 | time = 0.061s
schedules identical: True
speedup: 5122x
period 0: order =  12000, M =   6000, traded =  11071
period 1: order =  10000, M =   8000, traded =   9106
period 2: order =   9000, M =   8500, traded =   8171
period 3: order =   9000, M =   8750, traded =   8159
period 4: order =   9000, M =   8875, traded =   8153
period 5: order =   9000, M =   8938, traded =   8150
period 6: order =   9000, M =   8969, traded =   8148
period 7: order =   9000, M =   8985, traded =   8147
period 8: order =  10000, M =   9493, traded =   9026
period 9: order =  14000, M =  11747, traded =  12483
total traded = 90614


## Refinement check
Refine the grids ($n_m=n_r=n_n=n$, step size $N/n$) and track how the schedule, the true objective (exact integer simulation of the extracted schedule) and the DP's internal estimate d[0,0,N] change. All $n$ divide $N=100000$, so grids stay integer-valued. The true total should stabilize as the grid is refined, and the DP estimate should approach the true total (the interpolation error shrinks), assuming that the approximation is consistent.

In [13]:
# define grid resolutions to test for approximation refinement
grid_sizes = [50, 100, 200, 250, 500, 1000]
# initialize dictionaries and lists to store results and comparisons
results = {}
rows = []
# evaluate the vectorized DP at each grid resolution
for n in grid_sizes:
    # run the full vectorized DP pipeline using n intervals for each grid
    r = solve_vec(N, T, pi, alpha, n, n)
    # store the complete results for the current grid resolution
    results[n] = r
    # record grid spacing, objective values, runtime, and memory usage
    rows.append(dict(n=n, step=N // n, true_total=r['total'], dp_estimate=round(r['dp_estimate'], 1),
                     est_minus_true=round(r['dp_estimate'] - r['total'], 1),
                     runtime_s=round(r['t_total'], 2), value_table_MB=round(T * (n + 1)**2 * 8 / 1e6, 1)))
    # print the objective value and runtime for the completed resolution
    print(f"n={n:5d} done: total={r['total']}, time={r['t_total']:.2f}s")
# convert the refinement results into a pandas DataFrame
refine_df = pd.DataFrame(rows)
# display the comparison table across all tested grid resolutions
refine_df

n=   50 done: total=90605, time=0.02s
n=  100 done: total=90614, time=0.06s
n=  200 done: total=90613, time=0.37s
n=  250 done: total=90615, time=0.66s
n=  500 done: total=90616, time=6.62s
n= 1000 done: total=90617, time=69.32s


,n,step,true_total,dp_estimate,est_minus_true,runtime_s,value_table_MB
0,50,2000,90605,90609.2,4.2,0.02,0.2
1,100,1000,90614,90615.9,1.9,0.06,0.8
2,200,500,90613,90615.9,2.9,0.37,3.2
3,250,400,90615,90615.4,0.4,0.66,5.0
4,500,200,90616,90616.3,0.3,6.62,20.1
5,1000,100,90617,90617.8,0.8,69.32,80.2


In [14]:
sched_df = pd.DataFrame({f"n={n}": results[n]['schedule'] for n in grid_sizes},
                        index=[f"period {i}" for i in range(T)])
sched_df

,n=50,n=100,n=200,n=250,n=500,n=1000
period 0,12000,12000,11500,11600,11400,11700
period 1,10000,10000,10000,10000,9800,10100
period 2,10000,9000,9000,9600,9200,9100
period 3,8000,9000,9000,9200,8800,9000
period 4,8000,9000,9000,8800,9200,8800
period 5,10000,9000,9000,8800,9000,8900
period 6,8000,9000,9000,8800,9000,9100
period 7,10000,9000,9500,9200,9600,9200
period 8,10000,10000,10000,10000,10200,10000
period 9,14000,14000,14000,14000,13800,14100


In [15]:
best_n = max(grid_sizes, key=lambda n: results[n]['total'])
print("Change in schedule between consecutive refinement levels (L1 distance in shares):")
for a, b in zip(grid_sizes[:-1], grid_sizes[1:]):
    dist = int(np.abs(results[a]['schedule'] - results[b]['schedule']).sum())
    dtot = results[b]['total'] - results[a]['total']
    print(f"  n={a:5d} -> n={b:5d}:  L1 schedule change = {dist:6d},  change in true total = {dtot:+d}")
print(f"\nFinest grid n=1000: schedule = {results[1000]['schedule'].tolist()}, total = {results[1000]['total']}")

Change in schedule between consecutive refinement levels (L1 distance in shares):
  n=   50 -> n=  100:  L1 schedule change =   6000,  change in true total = +9
  n=  100 -> n=  200:  L1 schedule change =   1000,  change in true total = -1
  n=  200 -> n=  250:  L1 schedule change =   1800,  change in true total = +2
  n=  250 -> n=  500:  L1 schedule change =   2800,  change in true total = +1
  n=  500 -> n= 1000:  L1 schedule change =   2400,  change in true total = +1

Finest grid n=1000: schedule = [11700, 10100, 9100, 9000, 8800, 8900, 9100, 9200, 10000, 14100], total = 90617


In [16]:
final = results[1000]
print("Approximate optimal schedule (finest grid, n = 1000):")
print("  ", final['schedule'].tolist())
print("Total shares traded (exact simulation of that schedule):", final['total'])
print(f"Runtime (vectorized, n=1000): {final['t_total']:.2f} s")
print(f"\nBaseline n=100: loop {loop_time:.1f} s vs vectorized {res100['t_total']:.3f} s "
      f"({loop_time / res100['t_total']:.0f}x speedup), identical schedule and total {res100['total']}.")

Approximate optimal schedule (finest grid, n = 1000):
   [11700, 10100, 9100, 9000, 8800, 8900, 9100, 9200, 10000, 14100]
Total shares traded (exact simulation of that schedule): 90617
Runtime (vectorized, n=1000): 69.32 s

Baseline n=100: loop 310.4 s vs vectorized 0.061 s (5122x speedup), identical schedule and total 90614.


### Discussion

- **Correctness.** On identical grids ($n=20,40$), the vectorized and loop implementations produce identical value tables, schedules, and totals (maximum absolute difference 0.0). At $n=100$, both give the schedule *[12000, 10000, 9000, 9000, 9000, 9000, 9000, 9000, 10000, 14000]* and a total of **90614** shares. This confirms that vectorization changes how the recursion is computed without changing the approximation.

- **Speedup.** The loop implementation has complexity $O(T\cdot n_m\cdot n_r\cdot n_n)$, while the vectorized version precomputes *new_m*, *rew*, *lo*, and *w* and replaces repeated Python loops with NumPy operations. The measured speedups were approximately **462x** at $n=20$, **1124x** at $n=40$, and **5122x** at $n=100$ (310.4 seconds versus 0.061 seconds). Although both methods have the same theoretical complexity, vectorization greatly reduces runtime. Even at $n=1000$, the vectorized version finished in **69.32 seconds**.

- **Refinement.** As the grid increases from $n=50$ to $n=1000$, the simulated total stays between **90605 and 90617** shares, with successive changes of +9, -1, +2, +1, and +1. The difference between the DP estimate and simulated total also remains small (about **0.3–4.2 shares**), although this does not measure the error relative to the exact optimum. The schedule stays fairly consistent, with larger orders at the beginning and end and around 9000 shares per period in between. Some individual orders change by a few hundred shares, but the total barely changes, suggesting that several schedules perform similarly. At $n=1000$, the schedule is *[11700, 10100, 9100, 9000, 8800, 8900, 9100, 9200, 10000, 14100]*, trading **90617** shares.

- **Limitation / link to Task 1(b).** Each simulated total comes from a feasible integer schedule, so it provides a lower bound on the exact optimum. Our approximation restricts orders to a finite grid and uses linear interpolation for liquidity-memory states between grid points. We choose grid sizes that divide $N=100{,}000$ evenly so all orders are integers and sum to $N$. Once Task 1(b) is completed, we can add a *gap to exact* column to compare these approximate results with the exact optimal value.